In [1]:
#importing all the necessary libraries
import math
import networkx as nx
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import requests
import json
from datetime import datetime, timedelta
import gzip
import os
import time

In [2]:
#extracting the data from Polygon.io using the API
def fetch_forex_data_from_polygon(api_key, date):
    
    # Polygon.io forex endpoint for grouped daily data
    url = f"https://api.polygon.io/v2/aggs/grouped/locale/global/market/fx/{date}"
    
    params = {
        'adjusted': 'true',
        'apikey': api_key
    }
    
    print(f"Fetching forex data for {date}...")
    response = requests.get(url, params=params)
    
    if response.status_code != 200:
        print(f"Error: {response.status_code} - {response.text}")
        return None, response
    
    data = response.json()
    
    if 'results' not in data:
        print("No results found in the response")
        return None, response
    
    # Convert to DataFrame format similar to your existing structure
    records = []
    for result in data['results']:
        ticker = result['T']  # Ticker symbol like C:EURUSD
        
        # Skip if not a forex pair or doesn't match currency filter
        if not ticker.startswith('C:'):
            continue
            
        # if currencies:
        # Extract currency pair from ticker (e.g., C:EURUSD -> EUR, USD)
        pair = ticker[2:]  # Remove 'C:' prefix
        if len(pair) == 6:  # Standard format like EURUSD
            currency1, currency2 = pair[:3], pair[3:]
        else:
            # Handle other formats like EUR-USD
            parts = pair.replace('-', '').replace('/', '')
            if len(parts) == 6:
                currency1, currency2 = parts[:3], parts[3:]
            else:
                continue
                    
            # if currency1 not in currencies or currency2 not in currencies:
            #     continue
        
        # Create record with similar structure to your existing data
        record = {
            'ticker': ticker,
            'participant_timestamp': int(result['t']) * 1000000,  # Convert to nanoseconds
            # Add other fields if available
            'ask_price': result.get('a', 0),
            'bid_price': result.get('b', 0),
            'open': result.get('o', 0),
            'high': result.get('h', 0),
            'low': result.get('l', 0),
            'close': result.get('c', 0),
            'volume': result.get('v', 0)
        }
        records.append(record)
    
    df = pd.DataFrame(records)
    
    
    print(f"Fetched {len(df)} forex records")
    return df, response

POLYGON_API_KEY = "vSmtvpdrv8xGtZG7YYgSiPg3irGrpWbd"  

In [3]:
#extracting the data of each day for 3 months for targetted currencies
# taking the top most traded currencies and INR
df = pd.DataFrame()
date_range = pd.date_range(start="2026-06-01", end="2026-07-3", freq='D')

timedelta_minutes = timedelta(minutes=1)
for date in date_range:

    date_str = date.strftime("%Y-%m-%d") 
    daily_df, response = fetch_forex_data_from_polygon(
        api_key=POLYGON_API_KEY,
        date=date_str,  # YYYY-MM-DD format
    )
    if daily_df is None:
        #if error type is 429, wait for 60 seconds
        if response.status_code == 429:
            print("Rate limit exceeded. Waiting for 60 seconds...")
            time.sleep(60)
        
    if daily_df is not None:
        df = pd.concat([df, daily_df], ignore_index=True) 

Fetching forex data for 2026-06-01...
Fetched 1163 forex records
Fetching forex data for 2026-06-02...
Fetched 1204 forex records
Fetching forex data for 2026-06-03...
Fetched 1206 forex records
Fetching forex data for 2026-06-04...
Fetched 1168 forex records
Fetching forex data for 2026-06-05...
Fetched 1207 forex records
Fetching forex data for 2026-06-06...
Error: 429 - {"status":"ERROR","request_id":"1d663c8ac8e5d4cb3f5d24e7fc1eb001","error":"You've exceeded the maximum requests per minute, please wait or upgrade your subscription to continue. https://massive.com/pricing"}
Rate limit exceeded. Waiting for 60 seconds...
Fetching forex data for 2026-06-07...
Fetched 1207 forex records
Fetching forex data for 2026-06-08...
Fetched 1187 forex records
Fetching forex data for 2026-06-09...
Fetched 1206 forex records
Fetching forex data for 2026-06-10...
Fetched 1148 forex records
Fetching forex data for 2026-06-11...
Fetched 1168 forex records
Fetching forex data for 2026-06-12...
Fetche

In [4]:


# Save the combined DataFrame to a gzipped CSV
df.to_csv("data_set(1).csv.gz", compression='gzip', index=False)
df

,ticker,participant_timestamp,ask_price,bid_price,open,high,low,close,volume
0,C:ZAREUR,1780358399999000000,0,0,0.052708,0.052993,0.052052,0.052717,123113
1,C:SEKGBP,1780358399999000000,0,0,0.080323,0.080415,0.079053,0.079758,42306
2,C:CADHKD,1780358399999000000,0,0,5.678920,5.679485,5.655318,5.662163,121176
3,C:AUDHKD,1780358399999000000,0,0,5.625950,5.635160,5.589400,5.614730,139889
4,C:ZARAED,1780358399999000000,0,0,0.225884,0.226695,0.214094,0.225270,241088
...,...,...,...,...,...,...,...,...,...
29629,C:USDYER,1782950399999000000,0,0,238.250000,238.250000,238.250000,238.250000,3
29630,C:XPFUSD,1782950399999000000,0,0,0.009497,0.009497,0.009479,0.009479,3
29631,C:YERUSD,1782950399999000000,0,0,0.004186,0.004186,0.004185,0.004185,3
29632,C:USDSDG,1782950399999000000,0,0,597.000000,598.000000,597.000000,598.000000,3
